# YouTube Shorts Virality — Feature Engineering

**Goal:** predict, from a Short's first few hours, whether it will go viral.

**Setup carried over from `EDA_final.ipynb`**
- **Outcome:** views at **72 hours**, using the snapshot nearest 72h within ±2h.
- **Viral:** 72h views in the **top 10%**. The threshold is computed on the **training split only**, so the test set never influences it.
- **Snapshot rules:** `snapshot_as_of` for features (no look-ahead past the cutoff), `snapshot_nearest` for outcomes.

**Population:** all videos with a 72h outcome, from **every country**. Each video keeps its channel's `channel_country`, so the same split can later be filtered to US-only videos.

Data comes from `shorts-data-final.zip`, read directly without unzipping.

In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", 50)

DATA_ZIP = Path("shorts-data-final.zip")
ZIP_DATA_DIR = "shorts-data/data/"

OUTCOME_HOURS = 72     # outcome = views at 72h
OUTCOME_TOL = 2        # nearest snapshot within ± this many hours
VIRAL_QUANTILE = 0.90  # viral = top 10% of training-set 72h views
EARLY_WINDOW = 1       # an "as of h" feature is the latest snapshot with age in [h - 1, h]

TEST_SIZE = 0.20
RANDOM_STATE = 42

## Load the data from the zip

In [ ]:
def to_utc(values):
    # The collector writes ISO-8601 timestamps both with and without fractional seconds.
    return pd.to_datetime(values, utc=True, format="ISO8601", errors="coerce")

with zipfile.ZipFile(DATA_ZIP) as z:
    videos = pd.read_csv(z.open(ZIP_DATA_DIR + "videos.csv"))
    channels = pd.read_csv(z.open(ZIP_DATA_DIR + "channels.csv"))
    countries = pd.read_csv(z.open(ZIP_DATA_DIR + "channel_countries.csv"))
    snapshot_files = sorted(n for n in z.namelist()
                            if n.startswith(ZIP_DATA_DIR + "observations/") and n.endswith(".csv"))
    observations = pd.concat((pd.read_csv(z.open(n)) for n in snapshot_files), ignore_index=True)

for col in ["first_seen_at", "published_at"]:
    videos[col] = to_utc(videos[col])
for col in ["first_seen_at", "channel_created_at"]:
    channels[col] = to_utc(channels[col])
countries["country_checked_at"] = to_utc(countries["country_checked_at"])
observations["observed_at"] = to_utc(observations["observed_at"])
for col in ["age_hours", "view_count", "like_count", "comment_count"]:
    observations[col] = pd.to_numeric(observations[col], errors="coerce")
observations.sort_values(["video_id", "observed_at"], inplace=True, ignore_index=True)

DATA_SNAPSHOT_AT = observations["observed_at"].max()
print(f"DATA SNAPSHOT: {DATA_SNAPSHOT_AT:%Y-%m-%d %H:%M} UTC")
print(f"videos:        {videos.shape}")
print(f"channels:      {channels.shape}")
print(f"countries:     {countries.shape}")
print(f"observations:  {observations.shape} from {len(snapshot_files)} snapshot files")

## Snapshot selection rules

Copied from `EDA_final.ipynb` so features and outcomes are selected exactly the same way.

- `snapshot_as_of(obs, h, window)`: the **latest** snapshot with age between `h - window` and `h`. Used for **features**, so nothing observed after the cutoff can leak in.
- `snapshot_nearest(obs, h, tol)`: the snapshot **closest** to age `h` within `± tol`. Used for **outcomes**.

In [ ]:
def snapshot_as_of(obs, hour, window, required=("view_count",)):
    candidates = obs[obs["age_hours"].between(hour - window, hour)].dropna(subset=list(required))
    return (candidates.sort_values(["video_id", "age_hours"])
            .drop_duplicates("video_id", keep="last")
            .set_index("video_id"))

def snapshot_nearest(obs, hour, tol, required=("view_count",)):
    candidates = obs[obs["age_hours"].between(hour - tol, hour + tol)].dropna(subset=list(required))
    candidates = candidates.assign(_distance=(candidates["age_hours"] - hour).abs())
    return (candidates.sort_values(["video_id", "_distance", "age_hours"])
            .drop_duplicates("video_id", keep="first")
            .drop(columns="_distance")
            .set_index("video_id"))

## Modeling population: every video with a 72h outcome

A video needs a 72h snapshot to get a label. Videos without one are excluded: mostly videos published too recently to reach 72h before the data snapshot, plus a few with collection gaps around 72h. Country is attached but **not** filtered on.

In [ ]:
TARGET = f"views_{OUTCOME_HOURS}h"

outcome = snapshot_nearest(observations, OUTCOME_HOURS, OUTCOME_TOL)["view_count"].rename(TARGET)

countries_latest = countries.sort_values("country_checked_at").drop_duplicates("channel_id", keep="last")
population = (
    videos.merge(countries_latest[["channel_id", "channel_country"]], on="channel_id", how="left", validate="many_to_one")
    .merge(outcome, left_on="video_id", right_index=True, how="inner")
    .reset_index(drop=True)
)

hours_old = (DATA_SNAPSHOT_AT - videos["published_at"]).dt.total_seconds() / 3600
excluded = videos[~videos["video_id"].isin(population["video_id"])]
too_recent = (hours_old[excluded.index] < OUTCOME_HOURS + OUTCOME_TOL).sum()
print(f"videos with a {OUTCOME_HOURS}h outcome: {len(population):,} of {len(videos):,}")
print(f"excluded: {len(excluded):,} ({too_recent:,} too recent to reach {OUTCOME_HOURS}h, "
      f"{len(excluded) - too_recent:,} old enough but no snapshot within ±{OUTCOME_TOL}h)")
print(f"channels: {population['channel_id'].nunique():,}")
print("\ntop channel countries (NaN = channel did not state one):")
print(population["channel_country"].value_counts(dropna=False).head(8).to_string())

## 80/20 train/test split

Two choices beyond a plain random split:

1. **Grouped by channel.** Every video from a channel goes to the same side. Some channels have dozens of videos in the data (one has over 100). If a channel appeared on both sides, the model could do well on the test set by recognizing the channel (same subscriber count, same channel age) rather than learning what makes videos go viral. Grouping makes the test set behave like **new creators**.
2. **Stratified by 72h views.** Videos are binned into deciles of 72h views, and each side gets a similar mix, so the test set gets its fair share of top-10% videos.

`StratifiedGroupKFold` with 5 folds does both: one fold (≈20%) becomes the test set, the other four (≈80%) the training set. Because channels vary in size, the split is close to 80/20 but not exact.

In [ ]:
view_decile = pd.qcut(population[TARGET].rank(method="first"), 10, labels=False)

splitter = StratifiedGroupKFold(n_splits=round(1 / TEST_SIZE), shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(population, view_decile, groups=population["channel_id"]))
population["split"] = "train"
population.loc[test_idx, "split"] = "test"

train_videos = population[population["split"] == "train"].reset_index(drop=True)
test_videos = population[population["split"] == "test"].reset_index(drop=True)

shared_channels = set(train_videos["channel_id"]) & set(test_videos["channel_id"])
assert not shared_channels, "a channel appears in both train and test"
print(f"train: {len(train_videos):,} videos ({len(train_videos) / len(population):.1%}), "
      f"{train_videos['channel_id'].nunique():,} channels")
print(f"test:  {len(test_videos):,} videos ({len(test_videos) / len(population):.1%}), "
      f"{test_videos['channel_id'].nunique():,} channels")
print("channels in both: 0")

## Viral label: threshold from the training set only

The 90th percentile of 72h views is computed on **train**, then applied to both sets. Test videos never influence the threshold.

In [ ]:
VIRAL_THRESHOLD = train_videos[TARGET].quantile(VIRAL_QUANTILE)
for frame in (population, train_videos, test_videos):
    frame["viral"] = frame[TARGET] >= VIRAL_THRESHOLD

print(f"viral threshold (train 90th percentile): {VIRAL_THRESHOLD:,.0f} views at {OUTCOME_HOURS}h")
summary = pd.DataFrame({
    "videos": population.groupby("split").size(),
    "viral": population.groupby("split")["viral"].sum(),
    "viral rate": population.groupby("split")["viral"].mean().round(3),
    f"median {TARGET}": population.groupby("split")[TARGET].median(),
    "US channels": population.groupby("split")["channel_country"].apply(lambda s: (s == "US").sum()),
}).loc[["train", "test"]]
summary

## Save the split

Saves which side each video is on, so teammates (and later notebooks) use the exact same split. To restrict to US videos later, filter both sides on `channel_country == "US"`. Because the split is by channel, the filtered sets still never share a channel.

In [ ]:
SPLIT_PATH = Path("results") / "train_test_split.csv"
SPLIT_PATH.parent.mkdir(exist_ok=True)
population[["video_id", "channel_id", "channel_country", "split"]].to_csv(SPLIT_PATH, index=False)
print(f"saved {len(population):,} rows to {SPLIT_PATH}")

# Build features

Every feature below is a **per-video calculation**: it uses only that video's own snapshots, metadata and channel record. Nothing is learned from other videos, so the same `build_features` function is applied to train and test separately.

Steps that *are* learned from data (filling in missing values, scaling, capping outliers) are deferred to the modeling pipeline, where they will be fit on **train only**.

## How each feature is prepared

| Group | Feature | Preparation | Why |
|---|---|---|---|
| Early performance | `log_views_3h`, `log_views_6h` | Views from the latest snapshot at age 2–3h / 5–6h (`snapshot_as_of`), then `log1p` | Views span several orders of magnitude; the log puts a 1K→10K jump on the same footing as 100K→1M. `snapshot_as_of` guarantees nothing after the cutoff is used. |
| Early performance | `log_growth_3h_to_6h` | `log_views_6h − log_views_3h` | A ratio of 6h to 3h views: measures *how fast* a video is growing, separately from how many views it already has. |
| Engagement | `log_likes_6h`, `log_comments_6h` | `log1p` of counts at the 6h snapshot | Heavy-tailed counts, same reason as views. |
| Engagement | `like_rate_6h`, `comment_rate_6h` | likes / views and comments / views at 6h | How engaged viewers are, independent of audience size. |
| Engagement | `likes_hidden` | 1 if likes are missing at 6h but views exist | Creators can hide likes; the API then returns no value. Hidden likes stay **NaN** in the like features, and this flag records why. |
| Creator | `log_subscribers` | `log1p(subscriber_count)` | Heavy-tailed; recorded when the channel was first seen. |
| Creator | `log_channel_avg_views` | `log1p(total channel views / total channel videos)` | Historical popularity: how many views the channel's videos typically get. |
| Creator | `channel_age_days` | Days from channel creation to this video's publish time | Measured at publish time so it means the same thing for every video. |
| Content | `duration_seconds` | Unchanged | Already on a sensible scale (Shorts are at most 3 minutes). |
| Content | `title_len` | Number of characters in the title | |
| Content | `n_hashtags` | Number of `#hashtags` in the title and description | |
| Content | `emoji_count` | Number of emoji in the title (same pattern as `EDA_final.ipynb`) | |

**Missing values are left as NaN on purpose.** A video with no snapshot in the 2–3h or 5–6h window, or with hidden likes, gets NaN. The modeling pipeline decides how to fill these in, using training data only.

**Not used as features:** publish hour, `channel_id`, and anything from after the 6h cutoff, including the 72h outcome. Reasons are in the decision log at the end.

In [ ]:
EMOJI_PATTERN = "[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF☀-➿⬀-⯿]"

def build_features(videos_part):
    obs = observations[observations["video_id"].isin(videos_part["video_id"])]
    snap6 = snapshot_as_of(obs, 6, EARLY_WINDOW)[["view_count", "like_count", "comment_count"]].add_suffix("_6h")
    snap3 = snapshot_as_of(obs, 3, EARLY_WINDOW)[["view_count"]].add_suffix("_3h")
    ch = channels.set_index("channel_id")[["subscriber_count", "total_channel_views",
                                           "total_channel_videos", "channel_created_at"]]
    v = videos_part.set_index("video_id").join([snap6, snap3]).join(ch, on="channel_id")

    f = pd.DataFrame(index=v.index)
    # Early performance
    f["log_views_3h"] = np.log1p(v["view_count_3h"])
    f["log_views_6h"] = np.log1p(v["view_count_6h"])
    f["log_growth_3h_to_6h"] = f["log_views_6h"] - f["log_views_3h"]
    # Engagement at 6h
    f["log_likes_6h"] = np.log1p(v["like_count_6h"])  # NaN = creator hid likes
    f["log_comments_6h"] = np.log1p(v["comment_count_6h"])
    f["like_rate_6h"] = v["like_count_6h"] / v["view_count_6h"].replace(0, np.nan)
    f["comment_rate_6h"] = v["comment_count_6h"] / v["view_count_6h"].replace(0, np.nan)
    f["likes_hidden"] = (v["like_count_6h"].isna() & v["view_count_6h"].notna()).astype(int)
    # Creator
    f["log_subscribers"] = np.log1p(v["subscriber_count"])
    f["log_channel_avg_views"] = np.log1p(v["total_channel_views"] / v["total_channel_videos"].clip(lower=1))
    f["channel_age_days"] = (v["published_at"] - v["channel_created_at"]).dt.total_seconds() / 86400
    # Content
    titles = v["title"].fillna("")
    f["duration_seconds"] = v["duration_seconds"]
    f["title_len"] = titles.str.len()
    f["n_hashtags"] = (titles + " " + v["description"].fillna("")).str.count(r"#\w+")
    f["emoji_count"] = titles.str.count(EMOJI_PATTERN)
    return f

X_train = build_features(train_videos)
X_test = build_features(test_videos)
y_train = train_videos.set_index("video_id").loc[X_train.index, "viral"]
y_test = test_videos.set_index("video_id").loc[X_test.index, "viral"]

FEATURES = list(X_train.columns)
print(f"{len(FEATURES)} features")
print(f"X_train: {X_train.shape}, viral rate {y_train.mean():.3f}")
print(f"X_test:  {X_test.shape}, viral rate {y_test.mean():.3f}")

## Check the feature table

Missing values (computed on train) and a summary of each feature. Things to look for: unexpected missingness, negative channel ages, zero durations, or extreme values that might need capping in the pipeline.

In [ ]:
pd.DataFrame({
    "missing (train)": X_train.isna().sum(),
    "% missing (train)": X_train.isna().mean().mul(100).round(1),
    "% missing (test)": X_test.isna().mean().mul(100).round(1),
}).query("`missing (train)` > 0").sort_values("missing (train)", ascending=False)

In [ ]:
X_train.describe().T.round(3)

In [ ]:
print("negative channel ages:", (X_train["channel_age_days"] < 0).sum())
print("zero durations:       ", (X_train["duration_seconds"] == 0).sum())
print("views fell 3h→6h:     ", (X_train["log_growth_3h_to_6h"] < 0).sum())

## Feature decision log

**Dropped**
- **Publish hour (UTC).** Weak correlation with 72h views in the EDA. The data spans many countries, so a UTC hour is a different local time for different creators. Publish hour converted to local time with `channel_country` could be revisited later.
- **`channel_id`.** The split is grouped by channel, so test channels never appear in training; an ID feature could not help.
- **Anything observed after the 6h cutoff**, including the 72h outcome and `viral`. Using them would leak the answer.
- **Raw (unlogged) view, like and comment counts.** Replaced by their `log1p` versions.

**Engineered**
- **`log_growth_3h_to_6h`.** Separates growth speed from audience size, and is less redundant than feeding the model both raw view counts.
- **`like_rate_6h`, `comment_rate_6h`.** Engagement relative to audience size.
- **`likes_hidden`.** Records why like features are missing, since hidden likes reflect a creator setting, not random gaps.
- **`log_channel_avg_views`.** Historical channel popularity, the channel's typical views per video.
- **`channel_age_days`.** Channel age measured at each video's publish time.

# Baseline models

Two simple models, both trained on the training split and evaluated once on the test split:

1. **Linear regression → classification.** Predict a video's log 72h views, then call it viral if the prediction clears a cutoff.
2. **Logistic regression.** Predict the probability that a video goes viral directly.

Each model is a pipeline that fills in missing values (training-set median, plus a "was missing" flag for each column that has gaps) and then scales the features. Both steps are learned from **train only**.

**Metrics**
- **ROC AUC:** how well the model ranks viral videos above non-viral ones. 0.5 = random, 1.0 = perfect.
- **Average precision (AP):** like ROC AUC but focused on the viral class. Random guessing scores the viral rate (≈0.10), so this is the most informative single number when positives are rare.
- **Precision / recall at a cutoff:** of the videos called viral, the share that really were (precision); of the truly viral videos, the share that were caught (recall).
- **Accuracy is not reported.** Only 10% of videos are viral, so predicting "not viral" for everything would already be 90% accurate.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (r2_score, mean_absolute_error, roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

def preprocess(add_indicator=True):
    return [SimpleImputer(strategy="median", add_indicator=add_indicator), StandardScaler()]

def classification_row(name, y_true, score, called_viral):
    # One row of the results table: ranking metrics use the score, the rest use the yes/no calls.
    return {
        "model": name,
        "called viral": int(called_viral.sum()),
        "precision": precision_score(y_true, called_viral),
        "recall": recall_score(y_true, called_viral),
        "F1": f1_score(y_true, called_viral),
        "ROC AUC": roc_auc_score(y_true, score),
        "avg precision": average_precision_score(y_true, score),
    }

results = []
print(f"test set: {len(y_test):,} videos, {int(y_test.sum())} viral ({y_test.mean():.1%})")

## 1. Linear regression: predict 72h views

The target is `log1p(views_72h)`, since raw views are extremely heavy-tailed. **MAE** on the log scale converts to a typical multiplicative error: an MAE of 0.57 means predictions are typically off by a factor of about e^0.57 ≈ 1.8.

In [ ]:
y_train_log = np.log1p(train_videos.set_index("video_id").loc[X_train.index, TARGET])
y_test_log = np.log1p(test_videos.set_index("video_id").loc[X_test.index, TARGET])

linear = make_pipeline(*preprocess(), LinearRegression())
linear.fit(X_train, y_train_log)
pred_log_train = linear.predict(X_train)
pred_log_test = linear.predict(X_test)

mae = mean_absolute_error(y_test_log, pred_log_test)
print(f"R²:  {r2_score(y_test_log, pred_log_test):.3f}")
print(f"MAE: {mae:.3f} log views -> predictions typically off by a factor of {np.exp(mae):.2f}")

### From predicted views to a viral call

Two ways to choose the cutoff, both decided **without looking at the test set**:

- **Actual viral threshold:** call a video viral if its predicted 72h views reach the training-set threshold (`VIRAL_THRESHOLD`).
- **Top 10% of predictions:** call a video viral if its prediction is in the top 10% of *training-set predictions*.

Regression predictions are pulled toward the average: they vary less than real views do (compare the standard deviations printed below). The biggest videos' predictions therefore fall short of their true views, and the actual threshold misses some viral videos. The top-10% cutoff corrects for this.

In [ ]:
print(f"spread of log views — predicted: {pred_log_test.std():.2f}, actual: {y_test_log.std():.2f}")

cutoff_actual = np.log1p(VIRAL_THRESHOLD)
cutoff_top10 = np.quantile(pred_log_train, VIRAL_QUANTILE)
print(f"actual threshold:          {VIRAL_THRESHOLD:,.0f} views")
print(f"top 10% of predictions:    {np.expm1(cutoff_top10):,.0f} predicted views")

results.append(classification_row("Linear → actual threshold", y_test, pred_log_test, pred_log_test >= cutoff_actual))
results.append(classification_row("Linear → top 10% of predictions", y_test, pred_log_test, pred_log_test >= cutoff_top10))
pd.DataFrame(results).set_index("model").round(3)

## 2. Logistic regression: predict viral directly

Two cutoffs again: the default **50% probability**, and the **top 10% of training-set probabilities** so it is directly comparable with the linear model's top-10% rule.

In [ ]:
logistic = make_pipeline(*preprocess(), LogisticRegression(max_iter=1000))
logistic.fit(X_train, y_train)
prob_train = logistic.predict_proba(X_train)[:, 1]
prob_test = logistic.predict_proba(X_test)[:, 1]

prob_cutoff_top10 = np.quantile(prob_train, VIRAL_QUANTILE)
print(f"top-10% probability cutoff (from train): {prob_cutoff_top10:.3f}")

results.append(classification_row("Logistic → 50% probability", y_test, prob_test, prob_test >= 0.5))
results.append(classification_row("Logistic → top 10% of predictions", y_test, prob_test, prob_test >= prob_cutoff_top10))
print("confusion matrix at the top-10% cutoff (rows = actual not viral / viral, columns = predicted):")
print(confusion_matrix(y_test, prob_test >= prob_cutoff_top10))

## Baseline: early views alone

How much do the other features add beyond 6h views? This logistic regression uses only `log_views_6h`.

In [ ]:
views_only = make_pipeline(*preprocess(add_indicator=False), LogisticRegression(max_iter=1000))
views_only.fit(X_train[["log_views_6h"]], y_train)
prob_views_train = views_only.predict_proba(X_train[["log_views_6h"]])[:, 1]
prob_views_test = views_only.predict_proba(X_test[["log_views_6h"]])[:, 1]
results.append(classification_row("Views-only logistic → top 10%", y_test, prob_views_test,
                                  prob_views_test >= np.quantile(prob_views_train, VIRAL_QUANTILE)))

## Results

In [ ]:
results_table = pd.DataFrame(results).set_index("model")
results_table.style.format({c: "{:.3f}" for c in results_table.columns if c != "called viral"})

## ROC curves

Each curve sweeps the cutoff from "call everything viral" (top right) to "call nothing viral" (bottom left). At every cutoff it plots:

- **x: false positive rate.** The share of *non-viral* videos wrongly called viral.
- **y: true positive rate (recall).** The share of *viral* videos caught.

A curve that hugs the top-left corner catches most viral videos while raising few false alarms. The dashed diagonal is random guessing. **ROC AUC** is the area under each curve. The dot marks the logistic model's top-10% cutoff, the operating point used in the results table.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

INK, MUTED = "#3d3d3a", "#8a8a85"
curves = [  # (label, test-set score, color)
    ("Logistic, all features", prob_test, "#2a78d6"),
    ("Linear, all features", pred_log_test, "#eb6834"),
    ("Logistic, views at 6h only", prob_views_test, "#1baf7a"),
]

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--", label="Random guessing (AUC 0.500)")
for label, score, color in curves:
    fpr, tpr, _ = roc_curve(y_test, score)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC {roc_auc_score(y_test, score):.3f})")

# The logistic model's top-10% cutoff as a point on its curve
viral = y_test.to_numpy()
called = prob_test >= prob_cutoff_top10
point_fpr, point_tpr = called[~viral].mean(), called[viral].mean()
ax.plot(point_fpr, point_tpr, "o", ms=8, color="#2a78d6", mec="white", mew=2, zorder=5)
ax.annotate(f"Top-10% cutoff\ncatches {point_tpr:.0%} of viral videos,\n{point_fpr:.0%} false alarms",
            (point_fpr, point_tpr), xytext=(0.18, 0.55), color=INK, fontsize=9,
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=1))

ax.set(xlim=(0, 1), ylim=(0, 1.01), aspect="equal",
       xlabel="False positive rate (non-viral videos called viral)",
       ylabel="True positive rate (viral videos caught)")
ax.set_title(f"ROC curves on the test set ({len(y_test):,} videos, {int(y_test.sum())} viral)",
             loc="left", color=INK)
ax.grid(color="#e6e5e0", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(MUTED)
ax.tick_params(colors=MUTED, labelcolor=INK)
ax.legend(loc="lower right", frameon=False, fontsize=9)
fig.tight_layout()

ROC_PATH = Path("results") / "roc_curves.png"
fig.savefig(ROC_PATH, dpi=150)
plt.show()

## Which features matter?

Features are scaled, so coefficient sizes are comparable within a model. Positive = pushes toward more views / viral.

**Read these with care.** `log_growth_3h_to_6h` is exactly `log_views_6h − log_views_3h`, so those three features carry overlapping information and split credit between them arbitrarily. `likes_hidden` duplicates the automatic missing-likes flag. Predictions are unaffected, but individual coefficients for these features should not be interpreted on their own.

In [ ]:
feature_names = logistic[:-1].get_feature_names_out(X_train.columns)
coefficients = pd.DataFrame({
    "linear (log views)": linear[-1].coef_,
    "logistic (viral)": logistic[-1].coef_[0],
}, index=feature_names)
coefficients.sort_values("logistic (viral)", key=abs, ascending=False).round(3)

## Takeaways

- **Both models rank videos almost equally well**, so either works as a baseline. Linear regression gives an interpretable view-count forecast; logistic regression answers the yes/no question directly.
- **Choose the cutoff from training predictions, not the raw threshold.** Using the actual viral threshold on regression predictions misses viral videos, because regression predictions are pulled toward the average.
- **Early views carry most of the signal.** Compare the views-only row with the full models: the gap is what the creator, engagement and content features add.
- **Next steps:** a tree-based model (e.g. `HistGradientBoostingClassifier`) to capture interactions, cross-validation on the training set to choose cutoffs and settings, and a US-only comparison using `channel_country`.